# 03 — Feature Set 1
> **Chương 6.3** — One-hot encoding, log transform, feature ablation, Mutual Information


In [4]:

%run /content/Seminar-Pattern-Recognition/setup_colab.py

🔄 Repo đã có, pulling latest...
✅ Pull xong
📁 Working dir: /content/Seminar-Pattern-Recognition
🐍 src/ đã thêm vào sys.path
📦 Cài thư viện...
✅ Cài xong
📂 Thư mục data/ models/ reports/ sẵn sàng
🖥️  GPU: Tesla T4, 15360 MiB

🚀 Setup hoàn tất! Bắt đầu chạy notebook.


## 0. Import & Load

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import warnings; warnings.filterwarnings('ignore')

# Colab: setup_colab.py đã chạy trước → CWD = /content/Seminar-Pattern-Recognition
# Local: CWD có thể là notebooks/ → dùng .parent
PROJECT_ROOT = Path(os.getcwd())
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from feature_engineering import rare_category_grouper, log_transform_cols

RAW_DIR     = PROJECT_ROOT / 'data' / 'raw'
FEAT_DIR    = PROJECT_ROOT / 'data' / 'features'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FEAT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
SEED = 42

df = pd.read_csv(RAW_DIR / 'wikicities_raw.csv')
df = df.dropna(subset=['population']).copy()
df['log_population'] = np.log1p(df['population'])
y = df['log_population']
print(f' Loaded: {df.shape}')

## 1. Gộp categorical hiếm gặp → OTHER

In [ ]:
CAT_COLS = ['country_name', 'region_name']
MIN_FREQ = 10
df_fe = df.copy()
for col in CAT_COLS:
    before = df_fe[col].nunique()
    df_fe[col] = rare_category_grouper(df_fe[col], min_freq=MIN_FREQ)
    after = df_fe[col].nunique()
    print(f'{col}: {before} → {after} unique (gộp {before-after} nhóm hiếm vào OTHER)')

## 2. Log transform numeric features

In [ ]:
LOG_COLS = ['area', 'numTriples', 'abstractLen', 'populationDensity']
df_fe = log_transform_cols(df_fe, LOG_COLS)
print('Skewness trước → sau log transform:')
for col in LOG_COLS:
    if col in df.columns and f'log_{col}' in df_fe.columns:
        print(f'  {col:25s}: {df[col].skew():7.2f} → {df_fe[f"log_{col}"].skew():7.2f}')

## 3. One-Hot Encoding

In [ ]:
NUM_COLS = ['log_area','elevation','lat','lon','log_populationDensity',
            'foundingYear','numAttributes','log_numTriples','hasAbstract','log_abstractLen']

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = ohe.fit_transform(df_fe[CAT_COLS].fillna('UNKNOWN'))
df_cat = pd.DataFrame(cat_encoded, columns=ohe.get_feature_names_out(CAT_COLS), index=df_fe.index)

df_num = df_fe[[c for c in NUM_COLS if c in df_fe.columns]].copy()
X_set1 = pd.concat([df_num, df_cat], axis=1)

imputer = SimpleImputer(strategy='median')
X_set1 = pd.DataFrame(imputer.fit_transform(X_set1), columns=X_set1.columns)

print(f' Feature Set 1: {X_set1.shape[0]:,} rows × {X_set1.shape[1]} features')
print(f'   Numeric: {df_num.shape[1]} | One-hot: {df_cat.shape[1]}')

## 4. Baseline model

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
scores = -cross_val_score(rf, X_set1, y, cv=5, scoring='neg_root_mean_squared_error')
print(f'RMSE (log scale): {scores.mean():.4f} ± {scores.std():.4f}')

## 5. Feature Ablation Study

In [ ]:
ablation_groups = {
    'Without log_numTriples' : [c for c in X_set1.columns if c != 'log_numTriples'],
    'Without geo (lat/lon)'  : [c for c in X_set1.columns if c not in ['lat','lon']],
    'Without country OHE'    : [c for c in X_set1.columns if not c.startswith('country_name_')],
    'Without region OHE'     : [c for c in X_set1.columns if not c.startswith('region_name_')],
    'Numeric only'           : [c for c in X_set1.columns if c in NUM_COLS],
    'All features (baseline)': list(X_set1.columns),
}
baseline = None
results  = {}
for name, cols in ablation_groups.items():
    s = -cross_val_score(rf, X_set1[cols], y, cv=5, scoring='neg_root_mean_squared_error')
    results[name] = s.mean()
    print(f'{name:35s}: RMSE = {s.mean():.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
baseline_val = results['All features (baseline)']
colors = ['coral' if v > baseline_val else 'steelblue' for v in results.values()]
ax.barh(list(results.keys()), list(results.values()), color=colors)
ax.axvline(baseline_val, color='red', linestyle='--', label='Baseline')
ax.set_title('Feature Ablation (RMSE — thấp hơn = tốt hơn)')
ax.set_xlabel('RMSE'); ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'feature_ablation.png', bbox_inches='tight')
plt.show()

## 6. Mutual Information

In [ ]:
mi_scores = mutual_info_regression(X_set1, y, random_state=SEED)
mi_df = pd.Series(mi_scores, index=X_set1.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
mi_df.head(30).plot(kind='barh', ax=ax, color='teal')
ax.set_title('Top 30 features — Mutual Information với log(population)')
ax.set_xlabel('MI Score'); ax.invert_yaxis()
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'mutual_information_set1.png', bbox_inches='tight')
plt.show()
print(mi_df.head(10).to_string())

## 7. Lưu Feature Set 1

In [ ]:
X_set1['log_population'] = y.values
X_set1.to_csv(FEAT_DIR / 'feature_set1.csv', index=False)
mi_df.to_csv(FEAT_DIR / 'mi_scores_set1.csv', header=['mi_score'])
print(f' feature_set1.csv: {X_set1.shape}')
print('  Bước tiếp theo: 04_feature_set2.ipynb')